# Notebook setup

In [1]:
from pathlib import Path
import os

# show where the notebook is running
print("CWD before:", Path.cwd())

# Point to your package (adjust if needed)
# e.g. if your modules are under src/, add it to sys.path
import sys
sys.path.append(str(Path.cwd()))  # or Path("src").resolve()

print("CWD after:", Path.cwd())
# from data_processing.logging_utils import logger
# from data_processing.data_setup import create_data_directory

CWD before: /mnt/c/Users/Barrs/OneDrive - mail.tau.ac.il/MLHCproject/test2/mlhc_project/src
CWD after: /mnt/c/Users/Barrs/OneDrive - mail.tau.ac.il/MLHCproject/test2/mlhc_project/src


In [2]:
import pandas as pd
import pickle
from pathlib import Path
from typing import List, Tuple
import numpy as np

# from data_processing.integrated_data_preprocessor import IntegratedICUPreprocessor
# from cohort_data import get_cohort_hadm_ids_and_targets
# from logging_utils import logger
# from data_processing.data_setupd import create_data_directory

# Input CSV file paths
INITIAL_COHORT_CSV = "../csvs/initial_cohort.csv"    # Training/validation patient IDs
TEST_EXAMPLE_CSV = "../csvs/test_example.csv"        # Test set patient IDs

# Output directory for processed data
DATA_DIR = "data"

df_init = pd.read_csv(INITIAL_COHORT_CSV)
df_test = pd.read_csv(TEST_EXAMPLE_CSV)
# display(df_init.head()); display(df_test.head())
print("init shape:", df_init.shape, "test shape:", df_test.shape)

# For faster debug runs, sample a small subset (e.g., 100 patients)
INIT_SAMPLE_N = 200
TEST_SAMPLE_N = 50

# init_ids = df_init["subject_id"].astype(int).sample(min(INIT_SAMPLE_N, len(df_init)), random_state=42).tolist()
init_ids = df_init["subject_id"].astype(int).tolist()
test_ids = df_test["subject_id"].astype(int).sample(min(TEST_SAMPLE_N, len(df_test)), random_state=42).tolist()

print(len(init_ids), len(test_ids))

init shape: (32513, 1) test shape: (50, 1)
32513 50


## DB access sanity + cohort/targets only

In [3]:
import duckdb
from data_processing.data_extraction import DUCKDB_PATH  # or set your own path here

# db = Path("/mnt/c/Users/Barrs/My Drive/MIMIC-III/mimiciii.duckdb")
    # assert db.exists(), f"DB not found at {db}"
print("DUCKDB_PATH ->", DUCKDB_PATH)
# con = duckdb.connect(DUCKDB_PATH)
con = duckdb.connect(DUCKDB_PATH, read_only=True)

# con = duckdb.connect(str(db), read_only=True)

# sanity checks
print(con.execute("PRAGMA database_list").fetchdf())
print(con.execute("SHOW TABLES").fetchdf())

DUCKDB_PATH -> /mnt/c/Users/Barrs/My Drive/MIMIC-III/mimiciii.duckdb
   seq      name                                               file
0  570  mimiciii  /mnt/c/Users/Barrs/My Drive/MIMIC-III/mimiciii...
                  name
0           ADMISSIONS
1              CALLOUT
2           CAREGIVERS
3          CHARTEVENTS
4            CPTEVENTS
5       DATETIMEEVENTS
6        DIAGNOSES_ICD
7             DRGCODES
8                D_CPT
9      D_ICD_DIAGNOSES
10    D_ICD_PROCEDURES
11             D_ITEMS
12          D_LABITEMS
13            ICUSTAYS
14      INPUTEVENTS_CV
15      INPUTEVENTS_MV
16           LABEVENTS
17  MICROBIOLOGYEVENTS
18          NOTEEVENTS
19        OUTPUTEVENTS
20            PATIENTS
21       PRESCRIPTIONS
22  PROCEDUREEVENTS_MV
23      PROCEDURES_ICD
24            SERVICES
25           TRANSFERS


In [4]:
from data_processing.cohort_data import COHORT_SQL

# Register subject IDs as temporary table for SQL query
con.register("tmp_subject_ids", pd.DataFrame({"subject_id": init_ids}))

# Execute cohort SQL to get filtered admissions and target labels
df = con.execute(COHORT_SQL).fetchdf()
print(df.head())
print(df.shape)

# Extract admission IDs and target matrix
hadm_ids = df["hadm_id"].tolist()
targets = df[["mortality_event", "los_event", "readmission_event"]].reset_index(drop=True).values

   hadm_id  mortality_event  los_event  readmission_event
0   100003                0          0                  0
1   100006                0          1                  0
2   100007                0          1                  0
3   100009                0          0                  0
4   100010                0          0                  0
(22489, 4)


In [6]:
# Build a base table with everything we need
# returns all intermediate columns you need to check rules yourself (in Python):
#  subject_id, hadm_id, admittime, dischtime, deathtime, age, los_hours, has_chartevents_data, admission_rank, discharge_to_death_hours, discharge_to_readmission_hours, plus a computed helper died_within_54h.
#  It does not filter the cohort and does not output targets
base_sql = r"""
WITH ordered AS (
  SELECT
      a.subject_id::INTEGER            AS subject_id,
      a.hadm_id::INTEGER               AS hadm_id,
      a.admittime::TIMESTAMP           AS admittime,
      a.dischtime::TIMESTAMP           AS dischtime,
      a.deathtime::TIMESTAMP           AS deathtime,
      a.has_chartevents_data::INTEGER  AS has_chartevents_data,
      EXTRACT(year FROM AGE(a.admittime::TIMESTAMP, p.dob::TIMESTAMP))::INTEGER AS age,
      -- hospital LOS in hours
      EXTRACT(epoch FROM (a.dischtime::TIMESTAMP - a.admittime::TIMESTAMP)) / 3600.0 AS los_hours,
      -- death within first 54h of *admission* (Rule 5 exclusion)
      CASE
        WHEN a.deathtime IS NOT NULL
             AND EXTRACT(epoch FROM (a.deathtime::TIMESTAMP - a.admittime::TIMESTAMP)) / 3600.0 <= 54
        THEN 1 ELSE 0
      END AS died_within_54h,
      -- time to death after *discharge* (mortality target window)
      EXTRACT(epoch FROM (p.dod::TIMESTAMP - a.dischtime::TIMESTAMP)) / 3600.0 AS discharge_to_death_hours,
      -- time to next admission
      EXTRACT(epoch FROM (LEAD(a.admittime::TIMESTAMP) OVER (PARTITION BY a.subject_id ORDER BY a.admittime)
                          - a.dischtime::TIMESTAMP)) / 3600.0 AS discharge_to_readmission_hours,
      ROW_NUMBER() OVER (PARTITION BY a.subject_id ORDER BY a.admittime) AS admission_rank
  FROM admissions a
  JOIN patients  p ON a.subject_id = p.subject_id
  WHERE a.subject_id::INTEGER IN (SELECT subject_id FROM tmp_subject_ids)
)
SELECT *
FROM ordered
ORDER BY subject_id, admittime
"""
base_df = con.execute(base_sql).fetchdf()
base_df.head(10)
# print(base_df.shape)

,subject_id,hadm_id,admittime,dischtime,deathtime,has_chartevents_data,age,los_hours,died_within_54h,discharge_to_death_hours,discharge_to_readmission_hours,admission_rank
0,2,163353,2138-07-17 19:04:00,2138-07-21 15:48:00,NaT,1,0,92.733333,0,NaN,NaN,1
1,3,145834,2101-10-20 19:08:00,2101-10-31 13:58:00,NaT,1,76,258.833333,0,5410.033333,NaN,1
2,4,185777,2191-03-16 00:28:00,2191-03-23 18:41:00,NaT,1,47,186.216667,0,NaN,NaN,1
3,5,178980,2103-02-02 04:31:00,2103-02-04 12:15:00,NaT,1,0,55.733333,0,NaN,NaN,1
4,7,118037,2121-05-23 15:05:00,2121-05-27 11:57:00,NaT,1,0,92.866667,0,NaN,NaN,1
5,8,159514,2117-11-20 10:22:00,2117-11-24 14:20:00,NaT,1,0,99.966667,0,NaN,NaN,1
6,9,150750,2149-11-09 13:06:00,2149-11-14 10:15:00,2149-11-14 10:15:00,1,41,117.150000,0,-10.250000,NaN,1
7,11,194540,2178-04-16 06:18:00,2178-05-11 19:00:00,NaT,1,50,612.700000,0,4469.000000,NaN,1
8,12,112213,2104-08-07 10:15:00,2104-08-20 02:57:00,2104-08-20 02:57:00,1,72,304.700000,0,-2.950000,NaN,1
9,16,103251,2178-02-03 06:35:00,2178-02-05 10:51:00,NaT,1,0,52.266667,0,NaN,NaN,1


In [7]:
# Recreate the 5 rules in Python (ground truth inclusion)
MIN_AGE, MAX_AGE = 18, 89
MIN_LOS_HOURS = 54
MORTALITY_EVENT_HOURS = 720
LOS_EVENT_HOURS = 168
READMISSION_EVENT_HOURS = 720

def apply_rules(df):
    m1 = df["admission_rank"] == 1
    m2 = df["age"].between(MIN_AGE, MAX_AGE, inclusive="both")
    m3 = df["los_hours"] >= MIN_LOS_HOURS
    m4 = df["has_chartevents_data"] == 1
    m5 = df["died_within_54h"] == 0

    keep_mask = m1 & m2 & m3 & m4 & m5
    sizes = {
        "start": len(df),
        "rule1_first": int(m1.sum()),
        "rule2_age": int((m1 & m2).sum()),
        "rule3_los": int((m1 & m2 & m3).sum()),
        "rule4_chartevents": int((m1 & m2 & m3 & m4).sum()),
        "rule5_no_early_death": int(keep_mask.sum()),
    }
    # print(df[m3 & ~m5])
    return df[keep_mask].copy(), sizes

# print(base_df[base_df["died_within_54h"] == 1])
# print(base_df[base_df[base_df["los_hours"] >= MIN_LOS_HOURS]])
expected_cohort_df, sizes = apply_rules(base_df)
sizes, expected_cohort_df.shape


({'start': 41244,
  'rule1_first': 32513,
  'rule2_age': 25548,
  'rule3_los': 22927,
  'rule4_chartevents': 22493,
  'rule5_no_early_death': 22489},
 (22489, 12))

In [8]:
# Compute ground truth targets in Python
gt = expected_cohort_df.copy()

# Mortality: death within 30 days *after discharge* (using patients.dod vs dischtime)
gt["mortality_event"] = (gt["discharge_to_death_hours"] <= MORTALITY_EVENT_HOURS).astype(int)

# LOS>7 days target: NOTE this uses *hospital* LOS. If you intended ICU LOS, change source.
gt["los_event"] = (gt["los_hours"] > LOS_EVENT_HOURS).astype(int)

# Readmission within 30 days after discharge
gt["readmission_event"] = (gt["discharge_to_readmission_hours"] <= READMISSION_EVENT_HOURS).astype(int)

gt_targets = gt[["hadm_id","mortality_event","los_event","readmission_event"]].sort_values("hadm_id").reset_index(drop=True)
gt_targets.head()


,hadm_id,mortality_event,los_event,readmission_event
0,100003,0,0,0
1,100006,0,1,0
2,100007,0,1,0
3,100009,0,0,0
4,100010,0,0,0


In [9]:
from data_processing.cohort_data import get_cohort_hadm_ids_and_targets, COHORT_SQL
# Run COHORT_SQL and compare
sql_df = con.execute(COHORT_SQL).fetchdf()
sql_targets = sql_df[["hadm_id","mortality_event","los_event","readmission_event"]].sort_values("hadm_id").reset_index(drop=True)

# 1) Inclusion check: which HADM_IDs the SQL kept vs. Python rules
gt_hadm = set(gt_targets["hadm_id"].astype(int))
sql_hadm = set(sql_targets["hadm_id"].astype(int))

extra = sorted(sql_hadm - gt_hadm)    # in SQL but should be excluded
missing = sorted(gt_hadm - sql_hadm)  # expected by rules but not in SQL

print(f"GT size={len(gt_hadm)}  |  SQL size={len(sql_hadm)}")
print(f"Extra: {len(extra)}  Missing: {len(missing)}")
if extra:  display(base_df[base_df.hadm_id.isin(extra)].head(10))
if missing: display(base_df[base_df.hadm_id.isin(missing)].head(10))

# 2) Target equality for the intersection
common = sorted(gt_hadm & sql_hadm)
gt_common = gt_targets[gt_targets.hadm_id.isin(common)].sort_values("hadm_id").reset_index(drop=True)
sql_common = sql_targets[sql_targets.hadm_id.isin(common)].sort_values("hadm_id").reset_index(drop=True)

mismatch = (gt_common[["mortality_event","los_event","readmission_event"]].values !=
            sql_common[["mortality_event","los_event","readmission_event"]].values)
n_mismatch = int(mismatch.any(axis=1).sum())

print(f"Target mismatches on common HADM_IDs: {n_mismatch} / {len(common)}")
if n_mismatch:
    bad = gt_common.loc[mismatch.any(axis=1)]
    bad = bad.merge(sql_common, on="hadm_id", suffixes=("_gt","_sql"))
    display(bad.head(20))


GT size=22489  |  SQL size=22489
Extra: 0  Missing: 0
Target mismatches on common HADM_IDs: 0 / 22489


In [ ]:
# 1) Get the ORIGINAL outputs (what your pipeline returns today)
hadm_ids_orig, targets_orig = get_cohort_hadm_ids_and_targets(con, init_ids)
print("cohort #hadm:", len(hadm_ids_orig))
print("targets shape:", targets_orig.shape)
pd.DataFrame(targets_orig, columns=["mortality","los>7d","readm≤30d"]).head()

# 2) Convert them to a tidy DataFrame (so we can compare apples-to-apples)
orig_df = (
    pd.DataFrame({
        "hadm_id": hadm_ids_orig,
        "mortality_event": targets_orig[:, 0].astype(int),
        "los_event": targets_orig[:, 1].astype(int),
        "readmission_event": targets_orig[:, 2].astype(int),
    })
    # .sort_values("hadm_id")
    .reset_index(drop=True)
)

# 4) Cohort inclusion diffs (who is in vs. who should be in)
orig_set = set(orig_df["hadm_id"].astype(int))
gt_set   = set(gt_targets["hadm_id"].astype(int))

extra   = sorted(orig_set - gt_set)   # present in ORIGINAL but not in GT (over-inclusion)
missing = sorted(gt_set - orig_set)   # present in GT but not in ORIGINAL (over-filtering)

print(f"Original size = {len(orig_set)} | GT size = {len(gt_set)}")
print(f"Extra (in ORIGINAL, not GT): {len(extra)}")
print(f"Missing (in GT, not ORIGINAL): {len(missing)}")

if extra:
    display(base_df[base_df.hadm_id.isin(extra)][
        ["subject_id","hadm_id","admittime","dischtime","deathtime","age","los_hours","has_chartevents_data","admission_rank"]
    ].head(10))

if missing:
    display(base_df[base_df.hadm_id.isin(missing)][
        ["subject_id","hadm_id","admittime","dischtime","deathtime","age","los_hours","has_chartevents_data","admission_rank"]
    ].head(10))

# 5) Target label comparison on the intersection
common = sorted(orig_set & gt_set)
orig_common = orig_df[orig_df.hadm_id.isin(common)].reset_index(drop=True)
gt_common   = gt_targets[gt_targets.hadm_id.isin(common)].reset_index(drop=True)

# Vectorized mismatch check
cols = ["mortality_event","los_event","readmission_event"]
mismatch_mask = (orig_common[cols].values != gt_common[cols].values)
rows_with_any_mismatch = mismatch_mask.any(axis=1)
n_rows_bad = int(rows_with_any_mismatch.sum())

print(f"Target mismatches on common HADM_IDs: {n_rows_bad} / {len(common)}")
if n_rows_bad:
    bad = (
        orig_common.loc[rows_with_any_mismatch, ["hadm_id"] + cols]
        .merge(gt_common.loc[rows_with_any_mismatch, ["hadm_id"] + cols],
               on="hadm_id", suffixes=("_orig", "_gt"))
    )
    display(bad.head(20))

11:32:11.490 Started get_cohort_hadm_ids_and_targets
11:32:11.570 Finished get_cohort_hadm_ids_and_targets
cohort #hadm: 22489
targets shape: (22489, 3)
Original size = 22489 | GT size = 22489
Extra (in ORIGINAL, not GT): 0
Missing (in GT, not ORIGINAL): 0
Target mismatches on common HADM_IDs: 0 / 22489


In [11]:
# label distribution check (stratification sanity)
lab = pd.DataFrame(targets, columns=["mortality","los","readm"])
lab["combo"] = (lab["mortality"].astype(int).astype(str) +
                lab["los"].astype(int).astype(str) +
                lab["readm"].astype(int).astype(str))
lab["combo"].value_counts(normalize=True).rename("freq").to_frame()

,freq
combo,
010,0.443194
000,0.391436
110,0.074481
100,0.043443
011,0.026991
001,0.013829
111,0.004447
101,0.002179


## Static features only

In [12]:
from data_processing.static_data import (
    STATIC_SQL,                      # the query under test
    CATEGORICAL_COLUMNS, NUMERIC_COLUMNS, NUMERIC_COLUMNS_WITH_MISSING,
    BINARY_COLUMNS, STATIC_COLUMNS,
    HEIGHT_IN_ITEMIDS, HEIGHT_CM_ITEMIDS, WEIGHT_KG_ITEMIDS, WEIGHT_LB_ITEMIDS, WEIGHT_OZ_ITEMIDS,
    VASOPRESSOR_CV_ITEMIDS, VASOPRESSOR_MV_ITEMIDS,
    VENTILATION_PROCEDURE_ITEMIDS, VENTILATION_CHART_ITEMIDS,
    RRT_PROCEDURE_ITEMIDS, RRT_CHART_ITEMIDS,
    SEDATION_CV_ITEMIDS, SEDATION_MV_ITEMIDS,
    WINDOW_HOURS, IN_TO_CM_FACTOR, LB_TO_KG_FACTOR, ANTIBIOTIC_REGEX
)
from data_processing.static_data import get_static_data  # original function

base_hadm_ids = gt["hadm_id"].tolist()
base_targets = gt_targets[["mortality_event","los_event","readmission_event"]]
print("base cohort #hadm:", len(base_hadm_ids))
print("base targets shape:", base_targets.shape)
print("orig cohort #hadm:", len(hadm_ids_orig))
print("orig targets shape:", targets_orig.shape)

base cohort #hadm: 22489
base targets shape: (22489, 3)
orig cohort #hadm: 22489
orig targets shape: (22489, 3)


In [5]:
from data_processing.cohort_data import get_cohort_hadm_ids_and_targets
hadm_ids, targets = get_cohort_hadm_ids_and_targets(con, init_ids)
print("cohort #hadm:", len(hadm_ids))
print("targets shape:", targets.shape)

10:48:26.086 Started get_cohort_hadm_ids_and_targets
10:48:26.165 Finished get_cohort_hadm_ids_and_targets
cohort #hadm: 22489
targets shape: (22489, 3)


In [4]:
from data_processing.static_data import get_static_data, STATIC_COLUMNS

static_raw = get_static_data(con, hadm_ids)
print("static_raw shape:", static_raw.shape)

# peek as DataFrame with original column ordering
# from yourpkg.static_data import STATIC_COLUMNS
pd.DataFrame(static_raw, columns=STATIC_COLUMNS).head(10)

18:19:48.279 Started get_static_data
18:20:23.692 Finished get_static_data
static_raw shape: (22489, 18)


,admission_type,admission_location,insurance,language,religion,marital_status,ethnicity,gender,age,height,weight,hours_to_first_icu,received_vasopressor,recieved_mechanical_ventilation,received_rrt,received_sedation,received_antibiotic,reached_icu
0,EMERGENCY,EMERGENCY ROOM ADMIT,Private,ENGL,NOT SPECIFIED,SINGLE,WHITE,1,59,NaN,85.3,0,1,1,0,0,1,1
1,EMERGENCY,EMERGENCY ROOM ADMIT,Private,missing,NOT SPECIFIED,SINGLE,BLACK/AFRICAN AMERICAN,0,48,NaN,NaN,0,0,1,0,0,1,1
2,EMERGENCY,EMERGENCY ROOM ADMIT,Private,missing,JEWISH,MARRIED,WHITE,0,73,NaN,NaN,5,0,1,0,0,1,1
3,EMERGENCY,TRANSFER FROM HOSP/EXTRAM,Private,missing,CATHOLIC,MARRIED,WHITE,1,60,182.88,115.0,19,0,1,0,1,1,1
4,ELECTIVE,PHYS REFERRAL/NORMAL DELI,Private,ENGL,EPISCOPALIAN,MARRIED,WHITE,0,54,NaN,71.0,14,1,1,0,0,0,1
5,EMERGENCY,TRANSFER FROM HOSP/EXTRAM,Medicare,ENGL,CATHOLIC,MARRIED,WHITE,1,67,NaN,78.9,47,0,0,0,0,1,1
6,EMERGENCY,CLINIC REFERRAL/PREMATURE,Medicare,ENGL,PROTESTANT QUAKER,SINGLE,WHITE,1,55,137.16,49.7,0,0,1,0,1,1,1
7,EMERGENCY,EMERGENCY ROOM ADMIT,Medicaid,SPAN,UNOBTAINABLE,MARRIED,HISPANIC OR LATINO,1,54,NaN,NaN,<NA>,0,0,0,0,1,0
8,ELECTIVE,PHYS REFERRAL/NORMAL DELI,Medicare,ENGL,NOT SPECIFIED,MARRIED,UNKNOWN/NOT SPECIFIED,1,71,175.26,90.7,2,1,1,0,1,1,1
9,EMERGENCY,CLINIC REFERRAL/PREMATURE,Medicare,ENGL,CATHOLIC,SINGLE,WHITE,0,72,NaN,62.9,0,0,1,0,0,0,1


In [24]:
# --- Modular recomputation (from your cohort.py/queries.py) ---
import numpy as np
from cohort_constractions import (
    add_first_icu_intime, add_first_height, add_first_weight,
    add_received_vasopressor_flag, add_received_sedation_flag,
    add_was_mechanically_ventilated_flag, add_received_rrt_flag,
    add_received_antibiotic_flag
)

def static_via_steps(con, hadm_ids):
    # Build the SAME base fields as STATIC_SQL expects (using DuckDB so age is identical)
    con.register("tmp_hadm_ids", pd.DataFrame({"hadm_id": hadm_ids}))
    base = con.execute("""
        SELECT 
            a.hadm_id::INTEGER AS hadm_id,
            a.admission_type,
            a.admission_location,
            a.insurance,
            a.language,
            a.religion,
            a.marital_status,
            a.ethnicity,
            CASE WHEN p.gender = 'M' THEN 1 ELSE 0 END AS gender,
            EXTRACT(year FROM AGE(a.admittime::TIMESTAMP, p.dob::TIMESTAMP))::INTEGER AS age,
            a.admittime::TIMESTAMP AS admittime
        FROM admissions a
        JOIN patients p ON a.subject_id = p.subject_id
        WHERE a.hadm_id::INTEGER IN (SELECT hadm_id FROM tmp_hadm_ids)
        ORDER BY a.hadm_id
    """).fetchdf()

    # Standardize column names to lowercase
    base.columns = base.columns.str.lower()

    # Fill missing categoricals same as the production extractor
    for col in ["admission_type","admission_location","insurance","language","religion","marital_status","ethnicity"]:
        base[col] = base[col].fillna("missing")

    # Add ICU/time-window features using your modular helpers
    coh = base.copy()
    coh = add_first_height(con, hadm_ids, coh)                     # height_cm
    coh = add_first_weight(con, hadm_ids, coh)                     # weight_kg
    coh = add_first_icu_intime(con, hadm_ids, coh)                 # first_icu_intime
    coh = add_received_vasopressor_flag(con, hadm_ids, coh)        # received_vasopressor
    coh = add_received_sedation_flag(con, hadm_ids, coh)           # received_sedation
    coh = add_was_mechanically_ventilated_flag(con, hadm_ids, coh) # was_mechanically_ventilated
    coh = add_received_rrt_flag(con, hadm_ids, coh)                # received_rrt
    coh = add_received_antibiotic_flag(con, hadm_ids, coh)         # received_antibiotic

    # Adapt names/units so they MATCH STATIC_COLUMNS
    coh["height"] = coh["height_cm"]
    coh["weight"] = coh["weight_kg"]
    coh["reached_icu"] = coh["first_icu_intime"].notna().astype(int)
    # match the (typo) column name in STATIC_COLUMNS
    coh["recieved_mechanical_ventilation"] = coh["was_mechanically_ventilated"].astype(int)

    # # Keep exactly the STATIC schema
    # out = coh[["hadm_id"] + STATIC_COLUMNS].copy()
    # # Ensure numeric dtypes consistent
    # out["age"] = out["age"].astype(int)
    # for b in ["received_vasopressor","recieved_mechanical_ventilation","received_rrt","received_sedation","received_antibiotic","reached_icu"]:
    #     out[b] = out[b].fillna(0).astype(int)
    out = coh.copy()
    # print(out.head(20))
    return out

df_step = static_via_steps(con, base_hadm_ids)
display(df_step.head(10))

,hadm_id,admission_type,admission_location,insurance,language,religion,marital_status,ethnicity,gender,age,...,hours_to_first_icu,received_vasopressor,received_sedation,was_mechanically_ventilated,received_rrt,received_antibiotic,height,weight,reached_icu,recieved_mechanical_ventilation
0,100003,EMERGENCY,EMERGENCY ROOM ADMIT,Private,ENGL,NOT SPECIFIED,SINGLE,WHITE,1,59,...,0,1,0,0,0,1,NaN,85.3,1,0
1,100006,EMERGENCY,EMERGENCY ROOM ADMIT,Private,missing,NOT SPECIFIED,SINGLE,BLACK/AFRICAN AMERICAN,0,48,...,0,0,0,0,0,0,NaN,NaN,1,0
2,100007,EMERGENCY,EMERGENCY ROOM ADMIT,Private,missing,JEWISH,MARRIED,WHITE,0,73,...,5,0,0,0,0,1,NaN,NaN,1,0
3,100009,EMERGENCY,TRANSFER FROM HOSP/EXTRAM,Private,missing,CATHOLIC,MARRIED,WHITE,1,60,...,19,0,1,1,0,1,182.88,115.0,1,1
4,100010,ELECTIVE,PHYS REFERRAL/NORMAL DELI,Private,ENGL,EPISCOPALIAN,MARRIED,WHITE,0,54,...,14,1,1,0,0,0,NaN,71.0,1,0
5,100012,EMERGENCY,TRANSFER FROM HOSP/EXTRAM,Medicare,ENGL,CATHOLIC,MARRIED,WHITE,1,67,...,47,0,0,0,0,1,NaN,78.9,1,0
6,100016,EMERGENCY,CLINIC REFERRAL/PREMATURE,Medicare,ENGL,PROTESTANT QUAKER,SINGLE,WHITE,1,55,...,0,0,1,1,0,1,137.16,49.7,1,1
7,100021,EMERGENCY,EMERGENCY ROOM ADMIT,Medicaid,SPAN,UNOBTAINABLE,MARRIED,HISPANIC OR LATINO,1,54,...,<NA>,0,0,0,0,0,NaN,NaN,0,0
8,100024,ELECTIVE,PHYS REFERRAL/NORMAL DELI,Medicare,ENGL,NOT SPECIFIED,MARRIED,UNKNOWN/NOT SPECIFIED,1,71,...,2,1,1,1,0,0,175.26,90.7,1,1
9,100028,EMERGENCY,CLINIC REFERRAL/PREMATURE,Medicare,ENGL,CATHOLIC,SINGLE,WHITE,0,72,...,0,0,0,0,0,0,NaN,62.9,1,0


In [15]:
def static_via_sql(con, hadm_ids):
    # Register all temp tables exactly as the function does
    con.register("tmp_hadm_ids", pd.DataFrame({"hadm_id": hadm_ids}))
    con.register("tmp_height_in_itemids", pd.DataFrame({"itemid": HEIGHT_IN_ITEMIDS}))
    con.register("tmp_height_cm_itemids", pd.DataFrame({"itemid": HEIGHT_CM_ITEMIDS}))
    con.register("tmp_weight_kg_itemids", pd.DataFrame({"itemid": WEIGHT_KG_ITEMIDS}))
    con.register("tmp_weight_lb_itemids", pd.DataFrame({"itemid": WEIGHT_LB_ITEMIDS}))
    con.register("tmp_weight_oz_itemids", pd.DataFrame({"itemid": WEIGHT_OZ_ITEMIDS}))
    con.register("tmp_vaso_cv_itemids", pd.DataFrame({"itemid": VASOPRESSOR_CV_ITEMIDS}))
    con.register("tmp_vaso_mv_itemids", pd.DataFrame({"itemid": VASOPRESSOR_MV_ITEMIDS}))
    con.register("tmp_vent_proc_itemids", pd.DataFrame({"itemid": VENTILATION_PROCEDURE_ITEMIDS}))
    con.register("tmp_vent_chart_itemids", pd.DataFrame({"itemid": VENTILATION_CHART_ITEMIDS}))
    con.register("tmp_rrt_proc_itemids", pd.DataFrame({"itemid": RRT_PROCEDURE_ITEMIDS}))
    con.register("tmp_rrt_chart_itemids", pd.DataFrame({"itemid": RRT_CHART_ITEMIDS}))
    con.register("tmp_sed_cv_itemids", pd.DataFrame({"itemid": SEDATION_CV_ITEMIDS}))
    con.register("tmp_sed_mv_itemids", pd.DataFrame({"itemid": SEDATION_MV_ITEMIDS}))
    
    df = con.execute(STATIC_SQL).fetchdf()
    df.columns = df.columns.str.lower()
    # align categorical NA handling to the production function
    for col in CATEGORICAL_COLUMNS:
        df[col] = df[col].fillna("missing")
    # out = df[["hadm_id"] + STATIC_COLUMNS].copy()
    out = df.copy()
    return out

df_sql = static_via_sql(con, base_hadm_ids)
display(df_sql.head(10))

,hadm_id,admission_type,admission_location,insurance,language,religion,marital_status,ethnicity,gender,age,height,weight,hours_to_first_icu,reached_icu,recieved_mechanical_ventilation,received_rrt,received_vasopressor,received_sedation,received_antibiotic
0,100003,EMERGENCY,EMERGENCY ROOM ADMIT,Private,ENGL,NOT SPECIFIED,SINGLE,WHITE,1,59,NaN,85.3,0,1,1,0,1,0,1
1,100006,EMERGENCY,EMERGENCY ROOM ADMIT,Private,missing,NOT SPECIFIED,SINGLE,BLACK/AFRICAN AMERICAN,0,48,NaN,NaN,0,1,1,0,0,0,1
2,100007,EMERGENCY,EMERGENCY ROOM ADMIT,Private,missing,JEWISH,MARRIED,WHITE,0,73,NaN,NaN,5,1,1,0,0,0,1
3,100009,EMERGENCY,TRANSFER FROM HOSP/EXTRAM,Private,missing,CATHOLIC,MARRIED,WHITE,1,60,182.88,115.0,19,1,1,0,0,1,1
4,100010,ELECTIVE,PHYS REFERRAL/NORMAL DELI,Private,ENGL,EPISCOPALIAN,MARRIED,WHITE,0,54,NaN,71.0,14,1,1,0,1,1,0
5,100012,EMERGENCY,TRANSFER FROM HOSP/EXTRAM,Medicare,ENGL,CATHOLIC,MARRIED,WHITE,1,67,NaN,78.9,47,1,0,0,0,0,1
6,100016,EMERGENCY,CLINIC REFERRAL/PREMATURE,Medicare,ENGL,PROTESTANT QUAKER,SINGLE,WHITE,1,55,137.16,49.7,0,1,1,0,0,1,1
7,100021,EMERGENCY,EMERGENCY ROOM ADMIT,Medicaid,SPAN,UNOBTAINABLE,MARRIED,HISPANIC OR LATINO,1,54,NaN,NaN,<NA>,0,0,0,0,0,1
8,100024,ELECTIVE,PHYS REFERRAL/NORMAL DELI,Medicare,ENGL,NOT SPECIFIED,MARRIED,UNKNOWN/NOT SPECIFIED,1,71,175.26,90.7,2,1,1,0,1,1,1
9,100028,EMERGENCY,CLINIC REFERRAL/PREMATURE,Medicare,ENGL,CATHOLIC,SINGLE,WHITE,0,72,NaN,62.9,0,1,1,0,0,0,0


In [25]:
def compare_static_frames(df_sql, df_steps, atol=1e-9, max_show=15):
    # align rows
    left = df_sql.sort_values("hadm_id").reset_index(drop=True)
    right = df_steps.sort_values("hadm_id").reset_index(drop=True)

    # check cohort membership first
    left_ids = set(left.hadm_id.astype(int))
    right_ids = set(right.hadm_id.astype(int))
    extra = sorted(left_ids - right_ids)
    missing = sorted(right_ids - left_ids)
    print(f"#rows SQL={len(left)}  STEPS={len(right)}  |  extra={len(extra)}  missing={len(missing)}")
    if extra:
        display(left[left.hadm_id.isin(extra)].head(max_show))
    if missing:
        display(right[right.hadm_id.isin(missing)].head(max_show))

    # intersect for value comparison
    common = sorted(left_ids & right_ids)
    L = left[left.hadm_id.isin(common)].sort_values("hadm_id").reset_index(drop=True)
    R = right[right.hadm_id.isin(common)].sort_values("hadm_id").reset_index(drop=True)

    cat_cols = CATEGORICAL_COLUMNS
    num_cols = NUMERIC_COLUMNS + BINARY_COLUMNS

    # categorical exact compare
    cat_bad = []
    for c in cat_cols:
        ne = (L[c].astype(str).fillna("missing") != R[c].astype(str).fillna("missing"))
        if ne.any():
            cat_bad.append(c)
            print(f"[CAT] mismatch in '{c}': {int(ne.sum())}/{len(ne)} rows")
            display(pd.concat([L.loc[ne, ["hadm_id", c]].rename(columns={c: f"{c}_sql"}),
                               R.loc[ne, [c]].rename(columns={c: f"{c}_steps"})], axis=1).head(max_show))

    # numeric close compare
    num_bad = []
    for c in num_cols:
        lv = pd.to_numeric(L[c]).astype(float)
        rv = pd.to_numeric(R[c]).astype(float)
        same = np.isclose(lv.values, rv.values, equal_nan=True, atol=atol)
        if ~same.all():
            num_bad.append(c)
            ne = ~same
            print(f"[NUM] mismatch in '{c}': {int(ne.sum())}/{len(ne)} rows")
            display(pd.DataFrame({
                "hadm_id": L.loc[ne, "hadm_id"],
                f"{c}_sql": lv.loc[ne].values,
                f"{c}_steps": rv.loc[ne].values
            }).head(max_show))

    if not extra and not missing and not cat_bad and not num_bad:
        print("✅ Static features match perfectly (schema & values).")
    else:
        print("⚠️ Differences detected. See tables above.")

# hadm_ids you want to validate (e.g., from your cohort check)
df_sql  = static_via_sql(con, base_hadm_ids)
df_step = static_via_steps(con, base_hadm_ids)

compare_static_frames(df_sql, df_step)


#rows SQL=22489  STEPS=22489  |  extra=0  missing=0
[NUM] mismatch in 'recieved_mechanical_ventilation': 14102/22489 rows


,hadm_id,recieved_mechanical_ventilation_sql,recieved_mechanical_ventilation_steps
0,100003,1.0,0.0
1,100006,1.0,0.0
2,100007,1.0,0.0
4,100010,1.0,0.0
9,100028,1.0,0.0
13,100037,1.0,0.0
14,100041,1.0,0.0
15,100045,1.0,0.0
16,100050,1.0,0.0
17,100053,1.0,0.0


[NUM] mismatch in 'received_rrt': 7820/22489 rows


,hadm_id,received_rrt_sql,received_rrt_steps
14,100041,1.0,0.0
15,100045,1.0,0.0
16,100050,1.0,0.0
17,100053,1.0,0.0
18,100058,1.0,0.0
19,100059,1.0,0.0
24,100078,1.0,0.0
36,100142,1.0,0.0
40,100156,1.0,0.0
41,100160,1.0,0.0


[NUM] mismatch in 'received_antibiotic': 6157/22489 rows


,hadm_id,received_antibiotic_sql,received_antibiotic_steps
1,100006,1.0,0.0
7,100021,1.0,0.0
8,100024,1.0,0.0
10,100034,1.0,0.0
19,100059,0.0,1.0
23,100075,1.0,0.0
26,100087,1.0,0.0
35,100141,1.0,0.0
54,100230,1.0,0.0
56,100250,1.0,0.0


⚠️ Differences detected. See tables above.


In [6]:
from data_processing.static_data import (
    STATIC_SQL,                      # the query under test
    CATEGORICAL_COLUMNS, NUMERIC_COLUMNS, NUMERIC_COLUMNS_WITH_MISSING,
    BINARY_COLUMNS, STATIC_COLUMNS,
    HEIGHT_IN_ITEMIDS, HEIGHT_CM_ITEMIDS, WEIGHT_KG_ITEMIDS, WEIGHT_LB_ITEMIDS, WEIGHT_OZ_ITEMIDS,
    VASOPRESSOR_CV_ITEMIDS, VASOPRESSOR_MV_ITEMIDS,
    VENTILATION_PROCEDURE_ITEMIDS, VENTILATION_CHART_ITEMIDS,
    RRT_PROCEDURE_ITEMIDS, RRT_CHART_ITEMIDS,
    SEDATION_CV_ITEMIDS, SEDATION_MV_ITEMIDS,
    WINDOW_HOURS, IN_TO_CM_FACTOR, LB_TO_KG_FACTOR, ANTIBIOTIC_REGEX
)

In [7]:
def check_itemid_coverage(con, concept_name, search_pattern, current_ids, table='chartevents', limit=50):
    """
    Analyzes coverage for a specific clinical concept.
    
    Args:
        con: DuckDB connection
        concept_name: Name of concept (e.g. 'Weight')
        search_pattern: SQL LIKE pattern (e.g. '%weight%')
        current_ids: List of itemids currently in your script
        table: Table to search (chartevents, inputevents_mv, etc)
        limit: Number of rows to display (default 50). Set to None for all rows.
    """
    print(f"--- ANALYZING: {concept_name} ---")
    
    # Handle the LIMIT clause dynamically
    limit_clause = f"LIMIT {limit}" if limit else ""

    query = f"""
    WITH candidates AS (
        SELECT itemid, label 
        FROM d_items 
        WHERE label ILIKE '{search_pattern}'
    ),
    usage_counts AS (
        SELECT 
            c.itemid, 
            COUNT(*) as distinct_occurrences
        FROM {table} c
        JOIN candidates d ON c.itemid = d.itemid
        GROUP BY c.itemid
    )
    SELECT 
        u.itemid,
        d.label,
        u.distinct_occurrences,
        CASE WHEN u.itemid IN ({','.join(map(str, current_ids)) if current_ids else 'NULL'}) THEN 1 ELSE 0 END as is_included
    FROM usage_counts u
    JOIN candidates d ON u.itemid = d.itemid
    ORDER BY u.distinct_occurrences DESC
    {limit_clause};
    """
    
    df = con.execute(query).fetchdf()
    
    if df.empty:
        print("No matching items found in data.")
        return

    # Calculate coverage (based on the rows returned)
    total_rows = df['distinct_occurrences'].sum()
    included_rows = df[df['is_included'] == 1]['distinct_occurrences'].sum()
    coverage_pct = (included_rows / total_rows) * 100 if total_rows > 0 else 0.0
    
    print(f"Most Frequent Items matching '{search_pattern}' (Top {limit if limit else 'ALL'}):")
    
    # Use display() if available (Jupyter), otherwise print
    try:
        display(df)
    except NameError:
        print(df)
        
    print(f"Current list covers {coverage_pct:.2f}% of the data shown above.\n")

# Register admission IDs as temporary table for SQL query
con.register("tmp_hadm_ids", pd.DataFrame({"hadm_id": hadm_ids}))

find height and weight itemid

In [9]:
# Validate Weight
# Note: Combine your kg, lb, and oz lists for the check
all_weight_ids = WEIGHT_KG_ITEMIDS + WEIGHT_LB_ITEMIDS + WEIGHT_OZ_ITEMIDS
check_itemid_coverage(con, "Weight", "%weight%", all_weight_ids, table='chartevents')


--- ANALYZING: Weight ---
Most Frequent Items matching '%weight%' (Top 50):


,ITEMID,LABEL,distinct_occurrences,is_included
0,581,Previous WeightF,1641889,0
1,3723,Birth Weight (kg),435385,1
2,3580,Present Weight (kg),433003,1
3,3581,Present Weight (lb),432997,1
4,3582,Present Weight (oz),432996,1
5,3583,Previous Weight (kg),416621,0
6,3692,Weight Change (gms),416413,0
7,580,Previous Weight,146422,0
8,763,Daily Weight,47689,1
9,224639,Daily Weight,46452,1


Current list covers 41.27% of the data shown above.



In [10]:
# Validate Height
all_height_ids = HEIGHT_IN_ITEMIDS + HEIGHT_CM_ITEMIDS
check_itemid_coverage(con, "Height", "%height%", all_height_ids, table='chartevents')

--- ANALYZING: Height ---
Most Frequent Items matching '%height%' (Top 50):


,ITEMID,LABEL,distinct_occurrences,is_included
0,216,Height of Bed,106509,0
1,226707,Height,12015,1
2,226730,Height (cm),12015,1
3,1394,Height Inches,26,1


Current list covers 18.42% of the data shown above.



In [31]:
# Check units for the ambiguous 'Admit Wt' (762)
# This helps decide if 762 belongs in WEIGHT_LB_ITEMIDS or WEIGHT_KG_ITEMIDS
q_check_762 = """
SELECT valueuom, COUNT(*) as frequency
FROM chartevents 
WHERE itemid = 762 
GROUP BY valueuom
"""

print("--- Units for Item 762 (Admit Wt) ---")
display(con.execute(q_check_762).fetchdf())

--- Units for Item 762 (Admit Wt) ---


,VALUEUOM,frequency
0,None,11709
1,kg,29995


In [21]:
# Check the units (valueuom) for Item 226707
q_check_units = """
SELECT valuenum, valueuom, COUNT(*) as frequency
FROM chartevents
WHERE itemid = 226707
  AND valuenum IS NOT NULL
GROUP BY valuenum, valueuom
ORDER BY frequency DESC
LIMIT 10;
"""
display(con.execute(q_check_units).fetchdf())

,VALUENUM,VALUEUOM,frequency
0,70,Inch,1162
1,68,Inch,1020
2,66,Inch,933
3,67,Inch,927
4,65,Inch,926
5,64,Inch,875
6,72,Inch,827
7,69,Inch,795
8,62,Inch,748
9,63,Inch,671


In [25]:
height_ids = [920, 1394, 4187, 3486, 226707, 3485, 4188, 226730,]
weight_ids = [762, 763, 3723, 3580, 226512, 224639, 3581, 226531, 3582, 3693]

q_height_present = f"""
SELECT di.itemid, di.label, di.linksto, di.dbsource, COUNT(*) AS n
FROM chartevents ce
JOIN d_items di USING (itemid)
WHERE ce.itemid IN ({",".join(map(str, height_ids))})
GROUP BY 1,2,3,4
ORDER BY n DESC;
"""

q_weight_present = f"""
SELECT di.itemid, di.label, di.linksto, di.dbsource, COUNT(*) AS n
FROM chartevents ce
JOIN d_items di USING (itemid)
WHERE ce.itemid IN ({",".join(map(str, weight_ids))})
GROUP BY 1,2,3,4
ORDER BY n DESC;
"""

height_present = con.execute(q_height_present).fetchdf()
weight_present = con.execute(q_weight_present).fetchdf()

display(height_present)
display(weight_present)


,ITEMID,LABEL,LINKSTO,DBSOURCE,n
0,4188,Length in cm,chartevents,carevue,148996
1,4187,Length Calc Inches,chartevents,carevue,148995
2,920,Admit Ht,chartevents,carevue,41704
3,226707,Height,chartevents,metavision,12015
4,226730,Height (cm),chartevents,metavision,12015
5,3486,Length in Inches,chartevents,carevue,407
6,3485,Length Calc (cm),chartevents,carevue,388
7,1394,Height Inches,chartevents,carevue,26


,ITEMID,LABEL,LINKSTO,DBSOURCE,n
0,3723,Birth Weight (kg),chartevents,carevue,435385
1,3580,Present Weight (kg),chartevents,carevue,433003
2,3581,Present Weight (lb),chartevents,carevue,432997
3,3582,Present Weight (oz),chartevents,carevue,432996
4,763,Daily Weight,chartevents,carevue,47689
5,224639,Daily Weight,chartevents,metavision,46452
6,226531,Admission Weight (lbs.),chartevents,metavision,46255
7,762,Admit Wt,chartevents,carevue,41704
8,226512,Admission Weight (Kg),chartevents,metavision,22604
9,3693,Weight Kg,chartevents,carevue,994


find ventilation itemid

In [11]:
# Check Procedure Events
# Your current list: [225468, 224385, 224391]
check_itemid_coverage(con, "Ventilation Procedures", "%vent%", VENTILATION_PROCEDURE_ITEMIDS, table='procedureevents_mv', limit=20)
check_itemid_coverage(con, "Intubation Procedures", "%intub%", VENTILATION_PROCEDURE_ITEMIDS, table='procedureevents_mv', limit=20)

--- ANALYZING: Ventilation Procedures ---
Most Frequent Items matching '%vent%' (Top 20):


,ITEMID,LABEL,distinct_occurrences,is_included
0,225792,Invasive Ventilation,10749,1
1,225794,Non-invasive Ventilation,1411,0
2,225462,Interventional Radiology,907,0
3,226475,Intraventricular Drain Inserted,70,0


Current list covers 81.82% of the data shown above.

--- ANALYZING: Intubation Procedures ---
Most Frequent Items matching '%intub%' (Top 20):


,ITEMID,LABEL,distinct_occurrences,is_included
0,224385,Intubation,4514,1


Current list covers 100.00% of the data shown above.



In [12]:
# Check for "Ventilator Mode" or "Ventilator Type"
# Your current chart list is mixed, let's see what matches generic "Vent" strings
check_itemid_coverage(con, "Ventilator Settings/Mode", "%ventila%", VENTILATION_CHART_ITEMIDS, table='chartevents', limit=30)

--- ANALYZING: Ventilator Settings/Mode ---
Most Frequent Items matching '%ventila%' (Top 30):


,ITEMID,LABEL,distinct_occurrences,is_included
0,722,Ventilator Type,488468,1
1,720,Ventilator Mode,420719,1
2,223849,Ventilator Mode,241255,1
3,223848,Ventilator Type,204626,1
4,227565,Ventilator Tank #1,82167,0
5,227566,Ventilator Tank #2,82115,0
6,721,Ventilator No.,80002,0
7,3681,Ventilator Number,48527,0
8,3689,Vt [Ventilator],31662,0
9,5782,High Minute Ventilan,107,0


Current list covers 80.68% of the data shown above.



In [13]:
# Check for PEEP (Positive End-Expiratory Pressure)
# If you are missing high-frequency PEEP items, you are missing ventilation events.
check_itemid_coverage(con, "PEEP (Vent Proxy)", "%peep%", VENTILATION_CHART_ITEMIDS, table='chartevents', limit=20)

--- ANALYZING: PEEP (Vent Proxy) ---
Most Frequent Items matching '%peep%' (Top 20):


,ITEMID,LABEL,distinct_occurrences,is_included
0,220339,PEEP set,480563,1
1,506,PEEP Set,427050,1
2,505,PEEP,352328,1
3,3555,PEEP Alarm,212292,0
4,224700,Total PEEP Level,44275,0
5,60,Auto-PEEP Level,5275,1
6,686,Total PEEP Level,4544,0
7,437,Low Peep,4162,0
8,6924,MEASURED PEEP,17,0
9,6985,Sigh PEEP level,16,0


Current list covers 82.66% of the data shown above.



In [14]:
# Check for "O2 Delivery Device" items
# Note: You won't add these to your list immediately, but we need to see if "Ventilator" is a common value.
check_itemid_coverage(con, "O2 Delivery Device", "%delivery device%", VENTILATION_CHART_ITEMIDS, table='chartevents', limit=10)

--- ANALYZING: O2 Delivery Device ---
Most Frequent Items matching '%delivery device%' (Top 10):


,ITEMID,LABEL,distinct_occurrences,is_included
0,467,O2 Delivery Device,1155571,1
1,226732,O2 Delivery Device(s),523639,0
2,468,O2 Delivery Device#2,54729,0


Current list covers 66.64% of the data shown above.



In [32]:
# Check Procedure Events for "Ventilation"
check_itemid_coverage(con, "Ventilation (Procedures)", "%ventila%", VENTILATION_PROCEDURE_ITEMIDS, table='procedureevents_mv')

# Check Chart Events for "Ventilator Mode" or specific settings
# Common keywords: "Ventilator", "PEEP", "Tidal Volume"
check_itemid_coverage(con, "Ventilation (Chart)", "%ventila%", VENTILATION_CHART_ITEMIDS, table='chartevents')

# 1. Check Procedure Events (Extubation/Intubation)
check_itemid_coverage(con, "Ventilation (Proc)", "%vent%", VENTILATION_PROCEDURE_ITEMIDS, table='procedureevents_mv')

# 2. Check Chart Events (Ventilator Settings)
# We search for "PEEP" (Positive End-Expiratory Pressure) 
# because almost EVERY ventilated patient has a PEEP setting recorded.
# If you are missing the main PEEP item, you are missing ventilation data.
check_itemid_coverage(con, "Ventilation (Chart - PEEP)", "%peep%", VENTILATION_CHART_ITEMIDS, table='chartevents')

--- ANALYZING: Ventilation (Procedures) ---
Most Frequent Items matching '%ventila%' (Top 50):


,ITEMID,LABEL,distinct_occurrences,is_included
0,225792,Invasive Ventilation,10749,0
1,225794,Non-invasive Ventilation,1411,0


Current list covers 0.00% of the data shown above.

--- ANALYZING: Ventilation (Chart) ---
Most Frequent Items matching '%ventila%' (Top 50):


,ITEMID,LABEL,distinct_occurrences,is_included
0,722,Ventilator Type,488468,0
1,720,Ventilator Mode,420719,0
2,223849,Ventilator Mode,241255,0
3,223848,Ventilator Type,204626,0
4,227565,Ventilator Tank #1,82167,0
5,227566,Ventilator Tank #2,82115,0
6,721,Ventilator No.,80002,0
7,3681,Ventilator Number,48527,0
8,3689,Vt [Ventilator],31662,0
9,5782,High Minute Ventilan,107,0


Current list covers 0.00% of the data shown above.

--- ANALYZING: Ventilation (Proc) ---
Most Frequent Items matching '%vent%' (Top 50):


,ITEMID,LABEL,distinct_occurrences,is_included
0,225792,Invasive Ventilation,10749,0
1,225794,Non-invasive Ventilation,1411,0
2,225462,Interventional Radiology,907,0
3,226475,Intraventricular Drain Inserted,70,0


Current list covers 0.00% of the data shown above.

--- ANALYZING: Ventilation (Chart - PEEP) ---
Most Frequent Items matching '%peep%' (Top 50):


,ITEMID,LABEL,distinct_occurrences,is_included
0,220339,PEEP set,480563,1
1,506,PEEP Set,427050,1
2,505,PEEP,352328,1
3,3555,PEEP Alarm,212292,0
4,224700,Total PEEP Level,44275,0
5,60,Auto-PEEP Level,5275,1
6,686,Total PEEP Level,4544,0
7,437,Low Peep,4162,0
8,6924,MEASURED PEEP,17,0
9,6985,Sigh PEEP level,16,0


Current list covers 82.66% of the data shown above.



find Renal Replacement Therapy (RRT)

In [15]:
# 1. Check Procedures (MetaVision)
# Search for generic "Dialysis" and specific "CVVH" (Continuous Veno-Venous Hemofiltration)
check_itemid_coverage(con, "RRT Procedures (Dialysis)", "%dialysis%", RRT_PROCEDURE_ITEMIDS, table='procedureevents_mv', limit=20)
check_itemid_coverage(con, "RRT Procedures (CRRT/CVVH)", "%CVVH%", RRT_PROCEDURE_ITEMIDS, table='procedureevents_mv', limit=20)

--- ANALYZING: RRT Procedures (Dialysis) ---
Most Frequent Items matching '%dialysis%' (Top 20):


,ITEMID,LABEL,distinct_occurrences,is_included
0,225441,Hemodialysis,2064,0
1,225802,Dialysis - CRRT,1985,1
2,224270,Dialysis Catheter,1619,1
3,225805,Peritoneal Dialysis,226,1


Current list covers 64.98% of the data shown above.

--- ANALYZING: RRT Procedures (CRRT/CVVH) ---
No matching items found in data.


In [16]:
# 2. Check Chart Events (CareVue & MetaVision Settings)
# "Dialysis Type" is often the key field in older data
check_itemid_coverage(con, "RRT Chart (Dialysis)", "%dialysis%", RRT_CHART_ITEMIDS, table='chartevents', limit=20)
check_itemid_coverage(con, "RRT Chart (Hemodialysis)", "%hemodialysis%", RRT_CHART_ITEMIDS, table='chartevents', limit=20)

--- ANALYZING: RRT Chart (Dialysis) ---
Most Frequent Items matching '%dialysis%' (Top 20):


,ITEMID,LABEL,distinct_occurrences,is_included
0,152,Dialysis Type,59845,1
1,148,Dialysis Access Site,58864,0
2,149,Dialysis Access Type,58565,0
3,225323,Dialysis Catheter Site Appear,38910,0
4,151,Dialysis Site Appear,36071,0
5,225126,Dialysis patient,34174,1
6,227357,Dialysis Catheter Dressing Occlusive,33433,1
7,226118,Dialysis Catheter placed in outside facility,32078,0
8,227124,Dialysis Catheter Type,29546,0
9,150,Dialysis Machine,26581,0


Current list covers 29.72% of the data shown above.

--- ANALYZING: RRT Chart (Hemodialysis) ---
Most Frequent Items matching '%hemodialysis%' (Top 20):


,ITEMID,LABEL,distinct_occurrences,is_included
0,226499,Hemodialysis Output,1531,1


Current list covers 100.00% of the data shown above.



find sedation drugs

In [8]:
import re
import pandas as pd

SEDATION_CORE = [
    "propofol","diprivan", "dexmedetomidine","precedex", "ketamine","etomidate",   ## Hypnotic sedatives
    "lorazepam","diazepam", "alprazolam", "midazolam", "versed",  ## Benzodiazepines
    "barbiturates", "pentobarbital", "phenobarbital", "amobarbital", "secobarbital", "thiopental",  ## Barbiturates
    "clonidine",
]
OPIOIDS = ["fentanyl","remifentanil","morphine","hydromorphone", "sufentanil", "alfentanil"]

def _like_clause(col: str, terms: list[str]) -> str:
    terms = [t.lower() for t in terms]
    return " OR ".join([f"lower({col}) LIKE '%{t}%'" for t in terms])

def get_sedation_itemids_via_dictionary(con, include_opioids=False):
    terms = SEDATION_CORE + (OPIOIDS if include_opioids else [])
    like = _like_clause("d.label", terms) + " OR " + _like_clause("coalesce(d.abbreviation,'')", terms)

    # Helper to run a query and return a lowercased column DataFrame
    def _df(sql: str):
        df = con.execute(sql).fetchdf()
        df.columns = df.columns.str.lower()
        return df

    # Candidate IDs from D_ITEMS by linksto
    mv_df = _df(f"""
        SELECT DISTINCT d.itemid::INTEGER AS itemid, d.label AS label
        FROM d_items d
        WHERE d.linksto = 'inputevents_mv' AND ({like})
        ORDER BY d.itemid
    """)
    cv_df = _df(f"""
        SELECT DISTINCT d.itemid::INTEGER AS itemid, d.label AS label
        FROM d_items d
        WHERE d.linksto = 'inputevents_cv' AND ({like})
        ORDER BY d.itemid
    """)
    display(mv_df)
    display(cv_df)

    mv_ids = mv_df["itemid"].astype("Int64").dropna().astype(int).tolist()
    cv_ids = cv_df["itemid"].astype("Int64").dropna().astype(int).tolist()

    return cv_ids, mv_ids
    

# Example usage:
sed_cv_ids, sed_mv_ids =get_sedation_itemids_via_dictionary(con, include_opioids=True)
print("SEDATION_CV_ITEMIDS =", sed_cv_ids)
print("SEDATION_MV_ITEMIDS =", sed_mv_ids)


,itemid,label
0,221385,Lorazepam (Ativan)
1,221623,Diazepam (Valium)
2,221668,Midazolam (Versed)
3,221712,Ketamine
4,221744,Fentanyl
5,221833,Hydromorphone (Dilaudid)
6,222168,Propofol
7,225150,Dexmedetomidine (Precedex)
8,225154,Morphine Sulfate
9,225156,Pentobarbital


,itemid,label
0,30118,Fentanyl
1,30124,Midazolam
2,30126,Morphine Sulfate
3,30131,Propofol
4,30149,Fentanyl (Conc)
5,30150,Fentanyl Base
6,30151,Ketamine
7,30153,Morphine
8,30167,Precedex
9,30308,Fentanyl Drip


SEDATION_CV_ITEMIDS = [30118, 30124, 30126, 30131, 30149, 30150, 30151, 30153, 30167, 30308, 41516, 41733, 41962, 42062, 42407, 42596, 42669, 43136, 43387, 45476, 45520, 45563, 45573, 46301]
SEDATION_MV_ITEMIDS = [221385, 221623, 221668, 221712, 221744, 221833, 222168, 225150, 225154, 225156, 225942, 225972]


find Vasopressors itemid

In [22]:
def get_vaso_itemids_via_dictionary(con):
    # Keywords: Generic names + Brand names often found in older MIMIC data
    VASO_TERMS = [
        "norepinephrine", "levophed", 
        "epinephrine", "adrenalin", 
        "phenylephrine", "neo-synephrine", "neosynephrine",
        "vasopressin", "pitressin", 
        "dopamine", 
        "dobutamine", 
        "milrinone", "primacor"
    ]
    
    # Create the SQL OR clause
    terms_sql = " OR ".join([f"label ILIKE '%{t}%'" for t in VASO_TERMS])
    
    query = f"""
    SELECT 
        itemid, 
        label, 
        linksto,
        -- Count how often this item is actually used in the data
        (SELECT COUNT(*) FROM inputevents_cv WHERE itemid = d.itemid) + 
        (SELECT COUNT(*) FROM inputevents_mv WHERE itemid = d.itemid) as freq
    FROM d_items d
    WHERE (linksto = 'inputevents_cv' OR linksto = 'inputevents_mv')
      AND ({terms_sql})
    ORDER BY linksto, freq DESC
    """
    
    df = con.execute(query).fetchdf()
    
    # --- FIX: NORMALIZE COLUMNS TO LOWERCASE ---
    df.columns = df.columns.str.lower()
    
    # Split into CV and MV lists
    cv_df = df[df['linksto'] == 'inputevents_cv']
    mv_df = df[df['linksto'] == 'inputevents_mv']
    
    print("--- METAVISION (Newer System) ---")
    display(mv_df[mv_df['freq'] > 0]) # Only show items that have data
    
    print("\n--- CAREVUE (Older System) ---")
    display(cv_df[cv_df['freq'] > 0])
    
    # Return pure IDs for script
    return cv_df[cv_df['freq'] > 0]['itemid'].tolist(), mv_df[mv_df['freq'] > 0]['itemid'].tolist()

# Run it
vaso_cv_ids, vaso_mv_ids = get_vaso_itemids_via_dictionary(con)
print("\nSuggested VASOPRESSOR_CV_ITEMIDS =", vaso_cv_ids)
print("Suggested VASOPRESSOR_MV_ITEMIDS =", vaso_mv_ids)

--- METAVISION (Newer System) ---


,itemid,label,linksto,freq
15,221749,Phenylephrine,inputevents_mv,93571
16,221906,Norepinephrine,inputevents_mv,89697
17,221662,Dopamine,inputevents_mv,11389
18,221289,Epinephrine,inputevents_mv,6413
19,221986,Milrinone,inputevents_mv,6249
20,222315,Vasopressin,inputevents_mv,5648
21,221653,Dobutamine,inputevents_mv,2233



--- CAREVUE (Older System) ---


,itemid,label,linksto,freq
0,30128,Neosynephrine-k,inputevents_cv,554582
1,30120,Levophed-k,inputevents_cv,476971
2,30043,Dopamine,inputevents_cv,173745
3,30051,Vasopressin,inputevents_cv,165219
4,30125,Milrinone,inputevents_cv,132751
5,30119,Epinephrine-k,inputevents_cv,82889
6,30042,Dobutamine,inputevents_cv,66775
7,30307,Dopamine Drip,inputevents_cv,32971
8,30047,Levophed,inputevents_cv,22272
9,30127,Neosynephrine,inputevents_cv,14317



Suggested VASOPRESSOR_CV_ITEMIDS = ['30128', '30120', '30043', '30051', '30125', '30119', '30042', '30307', '30047', '30127', '30306', '30044', '30309', '42273', '42802']
Suggested VASOPRESSOR_MV_ITEMIDS = ['221749', '221906', '221662', '221289', '221986', '222315', '221653']


In [17]:
# Check CareVue (CV) Vasopressors
# Common drugs: Norepinephrine, Epinephrine, Dopamine, Phenylephrine, Vasopressin
check_itemid_coverage(con, "Norepinephrine (CV)", "%norepinephrine%", VASOPRESSOR_CV_ITEMIDS, table='inputevents_cv')
check_itemid_coverage(con, "Dopamine (CV)", "%dopamine%", VASOPRESSOR_CV_ITEMIDS, table='inputevents_cv')

# Check MetaVision (MV) Vasopressors
check_itemid_coverage(con, "Norepinephrine (MV)", "%norepinephrine%", VASOPRESSOR_MV_ITEMIDS, table='inputevents_mv')

--- ANALYZING: Norepinephrine (CV) ---
No matching items found in data.
--- ANALYZING: Dopamine (CV) ---
Most Frequent Items matching '%dopamine%' (Top 50):


,ITEMID,LABEL,distinct_occurrences,is_included
0,30043,Dopamine,173745,1
1,30307,Dopamine Drip,32971,1


Current list covers 100.00% of the data shown above.

--- ANALYZING: Norepinephrine (MV) ---
Most Frequent Items matching '%norepinephrine%' (Top 50):


,ITEMID,LABEL,distinct_occurrences,is_included
0,221906,Norepinephrine,89697,1


Current list covers 100.00% of the data shown above.



find antibiotics

In [18]:
# Your current regex (from your script)
# ANTIBIOTIC_REGEX = r'(amoxicillin|ampicillin|oxacillin|penicillin|piperacillin|tazobactam|zosyn|cefazolin|cefepime|ceftazidime|ceftriaxone|cefuroxime|meropenem|imipenem|ertapenem|vancomycin|amikacin|gentamicin|tobramycin|azithromycin|ciprofloxacin|levofloxacin|clindamycin|doxycycline|metronidazole|rifampin|daptomycin|linezolid)'

# Query to find "Suspicious" drugs that you missed
# We look for common antibiotic suffixes that DO NOT match your regex
q_missing_abx = f"""
SELECT 
    drug, 
    COUNT(*) as frequency
FROM prescriptions
WHERE (
       lower(drug) LIKE '%cillin%' 
    OR lower(drug) LIKE '%mycin%'
    OR lower(drug) LIKE '%oxacin%'
    OR lower(drug) LIKE '%cyclin%'
    OR lower(drug) LIKE '%penem%'
    OR lower(drug) LIKE '%cef%'
    OR lower(drug) LIKE '%sulf%'
)
AND NOT regexp_matches(lower(drug), '{ANTIBIOTIC_REGEX}')
GROUP BY drug
ORDER BY frequency DESC
LIMIT 20;
"""

print("--- Potential Antibiotics Missing from Regex ---")
display(con.execute(q_missing_abx).fetchdf())

--- Potential Antibiotics Missing from Regex ---


,DRUG,frequency
0,Magnesium Sulfate,90427
1,Morphine Sulfate,62134
2,Clopidogrel Bisulfate,5646
3,Ferrous Sulfate,5207
4,Atropine Sulfate,4977
5,Sodium Polystyrene Sulfonate,4702
6,NEO*PO*Ferrous Sulfate Elixir,2678
7,Morphine Sulfate IR,1754
8,Neomycin-Polymyxin-Bacitracin,1305
9,Readi-Cat 2 (Barium Sulfate 2% Suspension),1298


In [19]:
# Check which routes your regex is picking up
q_route_check = f"""
SELECT 
    route, 
    COUNT(*) as frequency
FROM prescriptions
WHERE regexp_matches(lower(drug), '{ANTIBIOTIC_REGEX}')
GROUP BY route
ORDER BY frequency DESC
LIMIT 15;
"""

print("--- Routes Captured by Current Regex ---")
display(con.execute(q_route_check).fetchdf())

--- Routes Captured by Current Regex ---


,ROUTE,frequency
0,IV,180158
1,PO,30213
2,PO/NG,10835
3,PB,1166
4,OU,879
5,NG,833
6,BOTH EYES,432
7,PR,301
8,OD,161
9,OS,146


16:30:33.865 Started get_static_data
16:31:10.835 Finished get_static_data
static_raw shape: (22489, 17)


,admission_type,admission_location,insurance,language,religion,marital_status,ethnicity,gender,age,height,weight,received_vasopressor,recieved_mechanical_ventilation,received_rrt,received_sedation,received_antibiotic,reached_icu
0,EMERGENCY,EMERGENCY ROOM ADMIT,Private,ENGL,NOT SPECIFIED,SINGLE,WHITE,1,59,NaN,85.3,1,1,0,0,1,1
1,EMERGENCY,EMERGENCY ROOM ADMIT,Private,missing,NOT SPECIFIED,SINGLE,BLACK/AFRICAN AMERICAN,0,48,NaN,NaN,0,1,0,0,1,1
2,EMERGENCY,EMERGENCY ROOM ADMIT,Private,missing,JEWISH,MARRIED,WHITE,0,73,NaN,NaN,0,1,0,0,1,1
3,EMERGENCY,TRANSFER FROM HOSP/EXTRAM,Private,missing,CATHOLIC,MARRIED,WHITE,1,60,182.88,115.0,0,1,0,1,1,1
4,ELECTIVE,PHYS REFERRAL/NORMAL DELI,Private,ENGL,EPISCOPALIAN,MARRIED,WHITE,0,54,NaN,71.0,1,1,0,0,0,1
5,EMERGENCY,TRANSFER FROM HOSP/EXTRAM,Medicare,ENGL,CATHOLIC,MARRIED,WHITE,1,67,NaN,78.9,0,0,0,0,1,1
6,EMERGENCY,CLINIC REFERRAL/PREMATURE,Medicare,ENGL,PROTESTANT QUAKER,SINGLE,WHITE,1,55,137.16,49.7,0,1,0,1,1,1
7,EMERGENCY,EMERGENCY ROOM ADMIT,Medicaid,SPAN,UNOBTAINABLE,MARRIED,HISPANIC OR LATINO,1,54,NaN,NaN,0,0,0,0,1,0
8,ELECTIVE,PHYS REFERRAL/NORMAL DELI,Medicare,ENGL,NOT SPECIFIED,MARRIED,UNKNOWN/NOT SPECIFIED,1,71,175.26,90.7,1,1,0,1,1,1
9,EMERGENCY,CLINIC REFERRAL/PREMATURE,Medicare,ENGL,CATHOLIC,SINGLE,WHITE,0,72,NaN,62.9,0,1,0,0,0,1


## Time-series features only

In [ ]:
from data_processing.timeseries_data import get_timeseries_data, TIMESERIES_COLUMNS

timeseries_data, timeseries_missingness = get_timeseries_data(con, hadm_ids)
# con.close()

print("ts_data:", timeseries_data.shape, "ts_miss:", timeseries_missingness.shape)
# tiny peek
N, H, F = timeseries_data.shape
timeseries_data[0, :3, :8]  # first patient, first 3 hours, first 8 features
pd.DataFrame(timeseries_data, columns=TIMESERIES_COLUMNS).head()

16:12:31.771 Started get_timeseries_data
16:12:40.256 Finished get_timeseries_data
ts_data: (61, 48, 216) ts_miss: (61, 48, 216)


ValueError: Must pass 2-d input. shape=(61, 48, 216)